# 18 MOVING: faults logged while moving

The blower message is a supervision artefact logged mostly at standstill. This notebook looks at what is left, and whether the approach extends to faults that develop while the locomotive is moving.

1. June onsets of the other messages by category and by motion at onset.
2. The primary XGBoost specification (17 measurement channels, default weighting) with five seeds, for two targets: other-message onsets logged while moving, and onsets in the traction and power conversion category. Seconds that are positive only because of onsets outside the target are excluded, as in `11_CHECK_BLOWER`.

The samples are small; results are reported with the number of positive seconds and onsets in the test partition. The category grouping here is an earlier, six-category version: the paper relies on the moving-state target, and counts traction messages with the seven categories of Table 1. Output: `OUTPUT/v4/rev_moving_*.csv`.

Paper: Section 4.5.

In [ ]:
import os, sys, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

V = "v4"            # data version: corrected log with the explicit exclusion rule (REV_REBUILD_v4)
env = os.environ.get
A1, A2 = "../ANALYSIS1", "../ANALYSIS2"
BUILD, DATA = env("REV_BUILDDIR", "build"), env("REV_DATADIR", "DATA")
CFG = dict(
    fault_log = f"{DATA}/37011FL{V}.csv",
    episodes  = f"{BUILD}/fault_episodes_{V}.csv",
    labelled  = f"{BUILD}/telemetry_labelled_{V}.pkl",
    july      = f"{BUILD}/telemetry_labelled_july_{V}.pkl",
    geo       = f"{A1}/DATA/37011/37011GEO.csv",
    outdir    = env("REV_OUTDIR", "OUTPUT") + f"/{V}",
    blower    = "AUX CAP MACH RM BLOW1 NOT OFF",
    june      = ("2025-06-01 00:00:00", "2025-06-30 23:59:59"),
    windows   = {"D5": (-5, 5), "W5": (-5, -1), "W60": (-60, -1), "W300": (-300, -1)},
    seeds     = [int(s) for s in env("REV_SEED_LIST", "42,1,2,3,4").split(",")],
    patience  = 100, test_frac = 0.20, val_frac = 0.20,
)
CATEGORY = {0: "Ventilation", 2: "Protection system", 3: "Protection system",
            5: "Traction and power conversion", 6: "Traction and power conversion",
            7: "Traction and power conversion", 9: "Braking and pneumatic",
            10: "Protection system", 11: "Protection system",
            12: "Traction and power conversion", 13: "Protection system",
            14: "Traction and power conversion", 15: "Auxiliary",
            17: "Power supply quality", 19: "Power supply quality",
            16: "Auxiliary", 8: "Protection system"}   # restored in v4: ASCU messages; Fire Alarm
NUMERIC = ["KMH", "DIST_M", "KV", "AMP", "KWH", "BV", "PRESSURE",
           "TRACTION_PCT", "BG1_KN", "BG2_KN", "BG1_DCLV", "BG2_DCLV"]
WHEEL_SLIP = ["Wheel_Slip_TM2", "Wheel_Slip_TM3", "Wheel_Slip_TM4",
              "Wheel_Slip_TM5", "Wheel_Slip_TM6"]
PRIMARY = NUMERIC + WHEEL_SLIP
os.makedirs(CFG["outdir"], exist_ok=True)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| data", V)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

def chrono_cut(y, frac):
    n = len(y); i = int(n * (1 - frac))
    while i < n - 1 and y[i] == 1 and y[i - 1] == 1:
        i += 1
    return i

def three_way_split(y):
    n = len(y); t = int(n * (1 - CFG["test_frac"]))
    while t < n - 1 and y[t] == 1:
        t += 1
    v = chrono_cut(y[:t], CFG["val_frac"])
    return np.arange(v), np.arange(v, t), np.arange(t), np.arange(t, n)

def xgb(y, seed, n_iter=None):
    spw = float((y == 0).sum() / max((y == 1).sum(), 1))
    kw = dict(n_estimators=600, early_stopping_rounds=CFG["patience"]) if n_iter is None else dict(n_estimators=n_iter)
    return XGBClassifier(learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                         eval_metric="aucpr", tree_method="hist", scale_pos_weight=spw,
                         random_state=seed, n_jobs=-1, **kw)

def fit_primary(X, y, seed):
    """Two-stage fit of 07_CLASSIFY (default class weighting). Returns the model fitted on the
    training partition, the test indices and the number of trees."""
    ia, iv, itr, ite = three_way_split(y)
    m1 = xgb(y[ia], seed).fit(X[ia], y[ia], eval_set=[(X[iv], y[iv])], verbose=False)
    n_iter = int(m1.best_iteration) + 1
    return xgb(y[itr], seed, n_iter).fit(X[itr], y[itr]), ite, n_iter

def pr(y, s):
    base = y.mean()
    ap = average_precision_score(y, s) if y.sum() > 0 else np.nan
    return dict(pr_auc=ap, baseline=base, lift=ap / base if base > 0 else np.nan, positives=int(y.sum()))

def coverage(ts, onsets, lo, hi):
    d = np.zeros(len(ts) + 1, dtype=np.int32)
    o = pd.DatetimeIndex(onsets).values.astype("datetime64[s]")
    a = np.searchsorted(ts, o + np.timedelta64(lo, "s"), side="left")
    b = np.searchsorted(ts, o + np.timedelta64(hi, "s"), side="right")
    np.add.at(d, a, 1); np.add.at(d, b, -1)
    return np.cumsum(d)[:-1]

def restrict(y0, ts, keep_onsets, other_onsets, h):
    """Positives produced only by onsets outside the kept set are excluded (-1), as in 11_CHECK_BLOWER."""
    lo, hi = CFG["windows"][h]
    cov_k = coverage(ts, keep_onsets, lo, hi)
    cov_o = coverage(ts, other_onsets, lo, hi)
    return np.where((y0 == 1) & (cov_o > 0) & (cov_k == 0), -1, y0)

## 1. Onsets by category and motion

In [ ]:
tel = pd.read_pickle(CFG["labelled"]).sort_values("DATETIME").reset_index(drop=True)
ts = tel["DATETIME"].values.astype("datetime64[s]")
ep = pd.read_csv(CFG["episodes"], parse_dates=["onset", "recovery"])
j0, j1 = (pd.Timestamp(x) for x in CFG["june"])
jun = ep[ep.onset.between(j0, j1)].copy()
jun["blower"] = jun.fault.eq(CFG["blower"])
jun["category"] = jun.cluster.map(CATEGORY).fillna("unassigned")
pos = pd.Series(np.arange(len(tel)), index=pd.DatetimeIndex(ts))
ii = pos.reindex(pd.DatetimeIndex(jun.onset)).values
ii = np.where(np.isnan(ii), pos.reindex(pd.DatetimeIndex(jun.onset) - pd.Timedelta(seconds=1)).values, ii)
jun["moving"] = [bool(tel["KMH"].iat[int(i)] > 0) if not np.isnan(i) else False for i in ii]
oth = jun[~jun.blower]
t = pd.crosstab(oth.category, oth.moving.map({True: "moving", False: "standing or no data"}), margins=True)
print("June onsets, other labels\n", t.to_string())
print("\nTraction and power conversion labels:",
      oth[oth.category == "Traction and power conversion"].fault.value_counts().to_dict())
t.to_csv(f"{CFG['outdir']}/rev_moving_counts.csv")

## 2. Fits

In [ ]:
TARGETS = {
    "other labels, moving": jun[~jun.blower & jun.moving].onset,
    "traction and power conversion": jun[jun.category == "Traction and power conversion"].onset,
}
X_all = tel[PRIMARY].astype("float32").fillna(0).values
rows = []
t_all = time.time()
for name, keep in TARGETS.items():
    rest = jun.onset[~jun.onset.isin(keep)]
    for h in env("REV_HORIZONS", "W5,W60,W300").split(","):
        y = restrict(tel[h].astype(int).values, ts, keep, rest, h)
        k = y != -1
        Xk, yk = X_all[k], y[k]
        for seed in CFG["seeds"]:
            m, ite, n_iter = fit_primary(Xk, yk, seed)
            r = pr(yk[ite], m.predict_proba(Xk[ite])[:, 1])
            tk = pd.DatetimeIndex(tel["DATETIME"].values[k][ite])
            r.update(target=name, horizon=h, seed=seed, trees=n_iter, onsets_total=len(keep),
                     onsets_in_test=int(pd.Series(keep).between(tk.min(), tk.max()).sum()))
            rows.append(r)
            print(f"{name:30s} {h:5s} seed {seed:2d}  PR-AUC {r['pr_auc']:.3f}  lift {r['lift']:.1f}x  "
                  f"test positives {r['positives']}  trees {n_iter}", flush=True)
res = pd.DataFrame(rows)
res.to_csv(f"{CFG['outdir']}/rev_moving_results.csv", index=False)
print(f"[{(time.time() - t_all) / 60:.1f} min]")

## 3. Summary over seeds

In [ ]:
s = res.groupby(["target", "horizon"]).agg(onsets=("onsets_total", "first"), onsets_in_test=("onsets_in_test", "first"), test_positives=("positives", "first"), baseline=("baseline", "first"), pr_auc_mean=("pr_auc", "mean"), pr_auc_sd=("pr_auc", "std"), lift_mean=("lift", "mean"))
print(s.round(4).to_string())
s.to_csv(f"{CFG['outdir']}/rev_moving_summary.csv")

___
**END**